# discriminator-classifier-head — ex2: PatchGAN-style classifier head (1x1 conv → per-patch sigmoid map)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `discriminator-classifier-head`. Running the final beacon cell reports progress against the `GAN: Discriminator classifier head` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Discriminator classifier head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`discriminator-classifier-head`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "discriminator-classifier-head"
DD_SUBTOPIC = "GAN: Discriminator classifier head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## PatchGAN-style classifier head — 1x1 conv → per-patch sigmoid map

Ex1 used `Flatten → Linear(D, 1) → Sigmoid` to produce ONE scalar real/fake
score per image. The deepening move is the **PatchGAN** head: instead of
collapsing to a single number, keep the spatial map and produce ONE score
PER PATCH using a `1x1 Conv2d(C, 1)` followed by sigmoid.

```python
self.head = nn.Conv2d(C_in, 1, kernel_size=1)   # 1x1 conv: per-pixel linear
# forward:
logits = self.head(features)         # (B, 1, H, W) — one logit per patch
probs  = t.sigmoid(logits)           # (B, 1, H, W) — per-patch real/fake
```

**Why 1x1 conv = per-position Linear.** A 1x1 conv with `C_in→1` is
exactly a Linear(C_in, 1) applied independently at every (h, w) cell of
the feature map. Shared weights across positions, identical to weight-tying
the linear head.

**Loss with PatchGAN.** Downstream BCE loss takes the per-patch sigmoid
map against a target map of all-ones (real) or all-zeros (fake) of
matching shape. The discriminator is forced to make decisions per local
receptive field rather than for the whole image — Pix2Pix's trick for
high-frequency texture realism.

### Exercise 2 — PatchGAN-style classifier head (1x1 conv → per-patch sigmoid map)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a 1x1 `Conv2d(C_in, 1)` followed by `sigmoid` to convert a discriminator feature map into a per-patch real/fake probability map of shape `(B, 1, H, W)` — the PatchGAN deepening of the Flatten→Linear scalar head.
> Keywords: patchgan, 1x1-conv, sigmoid, per-patch
> ```

**KCs targeted:** `1x1-conv-as-per-position-linear`, `spatial-sigmoid-output-map`

Implement `ex2_PatchHead`, an `nn.Module` for a PatchGAN-style discriminator head.

`__init__(in_channels: int)` must:
1. Call `super().__init__()`.
2. Create `self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)`.

`forward(x: Tensor) -> Tensor` where `x: (B, in_channels, H, W)`:
1. Apply `self.conv` to get logits of shape `(B, 1, H, W)`.
2. Return `t.sigmoid(logits)` — same shape, values in `(0, 1)`.

**No flatten, no Linear, no avg-pool.** The whole point is to keep the spatial dimension.

In [ ]:
class ex2_PatchHead(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        return t.sigmoid(self.conv(x))


<details><summary>Solution</summary>

```python
class ex2_PatchHead(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        return t.sigmoid(self.conv(x))
```

**1x1 conv = weight-tied Linear across spatial positions.** The (in_ch, 1, 1, 1) weight tensor reshapes to (1, in_ch) — exactly the same linear map applied independently at every (h, w) cell. Parameter count is `in_ch + 1` (weights + bias), independent of image size.

**Sigmoid OUTSIDE the conv, not as part of it.** `nn.Conv2d` doesn't take an activation; you compose `Conv2d → sigmoid` explicitly. Same pattern as `BCEWithLogitsLoss` users skip the sigmoid — but for inference you want probabilities.

**Why downstream loss changes too.** A PatchGAN trains with BCE against an `(B, 1, H, W)` target tensor (all-ones for real, all-zeros for fake). The reduction is `mean` over the patch dimension, so each patch contributes equally to the gradient.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()